In [ ]:
Merging the datasets into a complete csv.

In [2]:
import glob
import pandas as pd

files = glob.glob("Databases/*/sanitised.csv")
print(f"Found {len(files)} files")

dfs = []
for f in files:
    df = pd.read_csv(f, sep=",")  
    if not {"Uniprot_A", "Uniprot_B"}.issubset(df.columns):
        print(f"Skipping {f} — missing expected columns: {df.columns.tolist()}")
        continue
    df = df[["Uniprot_A", "Uniprot_B"]]
    df["source_file"] = f 
    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True)
print(f"Total rows before dedup: {len(merged)}")

merged[["Uniprot_A", "Uniprot_B"]] = merged.apply(
    lambda r: sorted([r["Uniprot_A"], r["Uniprot_B"]]), axis=1, result_type="expand"
)
merged = merged.drop_duplicates(subset=["Uniprot_A", "Uniprot_B"])
print(f"Total rows after dedup: {len(merged)}")

merged.to_csv("Databases/merged.csv", index=False)

Found 4 files
Total rows before dedup: 1843304
Total rows after dedup: 1161480


In [6]:
import pandas as pd
import networkx as nx
from collections import defaultdict

def create_graph(path_to_csv):
    dataframe = pd.read_csv(path_to_csv)
    G = nx.Graph()
    graph = defaultdict(set)
    
    for gene_a, gene_b in zip(dataframe['Uniprot_A'],dataframe['Uniprot_B']):
        G.add_nodes_from([gene_a,gene_b])
        if pd.isna(gene_a) or pd.isna(gene_b):
            continue
        if gene_a == gene_b:
            continue
        graph[gene_a].add(gene_b)
        graph[gene_b].add(gene_a)
        G.add_edge(gene_a,gene_b)
    return graph, G


In [4]:
from collections import deque

def BFS(source, targets, graph):
    if source in targets:
        return 0
    visited = set(source)
    queue = deque([(source,0)])
    while queue:
        node, dist = queue.popleft()
        for neighbour in list(graph[node]):
            if neighbour in visited:
                continue
            if neighbour in targets:
                return dist+1
            queue.append((neighbour, dist+1))
            visited.add(neighbour)
    return None

source = 'A0A024R2I8'
targets = ['Q9Y250','Q15796']
graph, _ = create_graph('Databases/merged.csv')

BFS(source,targets,graph)

2

In [5]:
def MS_BFS(source, targets, graph):
    if source in targets:
        return 0
    visited = set(targets)
    queue = deque((node,0) for node in targets)
    while queue:
        node, dist = queue.popleft()
        for neighbour in list(graph[node]):
            if neighbour in visited:
                continue
            if neighbour == source:
                return dist+1
            queue.append((neighbour, dist+1))
            visited.add(neighbour)
    return None



source = 'A0A024R2I8'
targets = ['Q9Y250','Q15796']
graph, _ = create_graph('Databases/merged.csv')


MS_BFS(source,targets,graph)


2

In [8]:
def BD_BFS(source, targets, graph):
    if source in targets:
        return 0
    dist_a = {node: 0 for node in targets}
    level_a = set(targets)

    dist_b = {source: 0}
    level_b = {source}

    while level_a and level_b:
        if len(level_a) <= len(level_b):
            level_a = next_level(graph, level_a, dist_a)
            cross = level_a & dist_b.keys()
        else:
            level_b = next_level(graph, level_b, dist_b)
            cross = level_b & dist_a.keys()
        if cross:
            return min(dist_a[node] + dist_b[node] for node in cross)
    return None

def next_level(graph, level, dist):
    next_level = set()
    for node in level:
        for neighbour in list(graph[node]):
            if neighbour not in dist:
                dist[neighbour] = dist[node]+1
                next_level.add(neighbour)
    return next_level

source = 'A0A024R2I8'
targets = ['Q9Y250','Q15796']
graph, G = create_graph('Databases/merged.csv')


BD_BFS(source,targets,graph)


2